In [ ]:
# Ensure the proteus package is importable from the notebooks/ subdirectory
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

# Lab 3: Context Engineering & Web Search

**Context Window Management, Summarization, and Tavily Search**

While memory engineering focuses on *what to store and retrieve*, context engineering
focuses on *how to manage what's in the context window right now*. In this lab you'll
use the context utilities from `proteus.context` and `proteus.tools`, register
agent-callable tools for summarization and web search, and test them end-to-end.

## Task 1: Restore credentials and rebuild state from Lab 2

We need the database connection, MemoryManager, Toolbox, and OpenAI client
that were created in Lab 2. This cell rebuilds them from the `proteus` modules.

In [ ]:
import os, getpass

# Restore credentials
%store -r adb_dsn vector_user vector_password
os.environ["PROTEUS_DB_DSN"] = adb_dsn
os.environ["PROTEUS_DB_PASSWORD"] = vector_password
os.environ["PROTEUS_DB_USER"] = vector_user

# OpenAI key
openai_key = getpass.getpass("OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_key

In [ ]:
from proteus.db import connect_to_oracle
from proteus.vector_store import get_embedding_model, create_all_vector_stores
from proteus.memory_manager import MemoryManager
from proteus.toolbox import Toolbox
from proteus import config
from openai import OpenAI

# Reconnect
vector_conn = connect_to_oracle()
embedding_model = get_embedding_model()
stores = create_all_vector_stores(vector_conn, embedding_model)

memory_manager = MemoryManager(
    conn=vector_conn,
    conversation_table=config.CONVERSATIONAL_TABLE,
    knowledge_base_vs=stores["knowledge_base_vs"],
    workflow_vs=stores["workflow_vs"],
    toolbox_vs=stores["toolbox_vs"],
    entity_vs=stores["entity_vs"],
    summary_vs=stores["summary_vs"],
    tool_log_table=config.TOOL_LOG_TABLE,
)

client = OpenAI()
toolbox = Toolbox(memory_manager=memory_manager, llm_client=client, embedding_model=embedding_model)
print("✅ State rebuilt — MemoryManager and Toolbox ready")

## Task 2: Context window usage calculator

This utility estimates how much of the context window is being used.
Proteus checks this to decide whether compaction is needed.

In [ ]:
from proteus.context import calculate_context_usage

# Test with a sample context string
sample_context = "This is a sample context. " * 500
usage = calculate_context_usage(sample_context)
print(f"Estimated tokens: {usage['tokens']}")
print(f"Max tokens: {usage['max']}")
print(f"Usage: {usage['percent']}%")

## Task 3: Context summarizer

When the context grows large, we compress it into a summary stored in Summary Memory.
The context window gets a compact pointer instead of the full content.

In [ ]:
from proteus.context import summarise_context_window

# Summarize a chunk of text
test_content = memory_manager.read_knowledge_base("transformer architectures")
result = summarise_context_window(test_content, memory_manager, client)

print(f"Summary ID: {result['id']}")
print(f"Description: {result['description']}")
print(f"Summary:\n{result['summary']}")

## Task 4: Register all agent tools

`proteus.tools.init_tools()` registers the summary tools, paper lookup,
and web search (Tavily) into the Toolbox in one call.

In [ ]:
# Tavily key is optional — web search will be skipped if not provided
tavily_key = getpass.getpass("Tavily API Key (press Enter to skip): ")
if tavily_key:
    os.environ["TAVILY_API_KEY"] = tavily_key

In [ ]:
from proteus.tools import init_tools

init_tools(memory_manager, toolbox, client, tavily_api_key=tavily_key or None)

## Task 5: Test tool retrieval

Let's verify Proteus can find tools semantically.

In [ ]:
import pprint

# Should find search_tavily (if registered) or other relevant tools
retrieved = memory_manager.read_toolbox("Search the internet for recent research papers")
pprint.pprint(retrieved)

In [ ]:
# Should find expand_summary or summarize tools
retrieved = memory_manager.read_toolbox("compact the conversation context")
pprint.pprint([t["function"]["name"] for t in retrieved])

## Task 6: Test the search-and-store pattern (if Tavily is available)

When Proteus calls `search_tavily()`, results are automatically persisted
to the knowledge base — so future queries find them without another API call.

In [ ]:
from proteus.tools import search_tavily

if os.environ.get("TAVILY_API_KEY"):
    results = search_tavily("recent advances in AI agent memory 2026")
    print(f"Found {len(results)} results")
    for r in results[:2]:
        print(f"  • {r.get('title', 'N/A')}")

    # Verify it was stored in the knowledge base
    print("\n--- Knowledge base check ---")
    print(memory_manager.read_knowledge_base("AI agent memory 2026")[:500])
else:
    print("⏭️ Tavily not configured — skipping web search test")
    print("💡 Set TAVILY_API_KEY to enable. Alternative: duckduckgo-search package.")

✅ **Lab 3 complete!** You now have:
- Context usage monitoring
- LLM-powered summarization with summary memory storage
- Agent-callable tools: `expand_summary`, `summarize_conversation`, `summarize_and_store`
- Web search via Tavily with automatic knowledge base persistence
- All tools registered in the semantic Toolbox for retrieval

**Next up:** Lab 4 — Agent Execution & Evaluation